# Image Type Profiling

Sample-based profiling for mixed calligraphy source images. This notebook does not run normal preprocessing, model training, evaluation, or dataset splitting.

## Run Settings

`RUN_FULL = False`, `SAMPLE_SIZE = 300`. The sampler stratifies by bundle and target style, then supplements darker images to expose black-background/rubbing cases.

In [1]:

from __future__ import annotations

import math
import random
import re
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont, ImageOps, UnidentifiedImageError
from scipy import ndimage

# Config
ROOT = Path('/Users/lainantian/Developer/Caligraphy')
OUTPUT_DIR = ROOT / 'image_type_profile'
SAMPLES_DIR = OUTPUT_DIR / 'samples'
RUN_FULL = False
SAMPLE_SIZE = 300
RANDOM_SEED = 42
DARK_POOL_SIZE = 3000
DARK_SUPPLEMENT_SIZE = 84

BUNDLES = [
    'common_c_bundle',
    'common_g_bundle 2',
    'common_h_bundle',
    'common_i_bundle',
    'common_j_bundle',
    'less_common_b_bundle',
]
TARGET_STYLE_LABELS = {
    '楷书': '楷書',
    '篆书': '篆書',
    '草书': '草書',
    '章草': '草書',
    '行书': '行書',
    '隶书': '隸書',
}
IMAGE_TYPES = [
    'clean_white',
    'yellow_paper',
    'gray_texture',
    'black_clean',
    'black_gray_ink',
    'black_damaged',
    'black_noisy',
    'rubbing_noisy',
    'catalog_or_invalid',
    'unknown',
]
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
VECTOR_HINTS = ['如果不满意这么大', '矢量部分', '找到原大的']

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


def bundle_prefix(bundle: str) -> str:
    if bundle == 'less_common_b_bundle':
        return 'less_common_b'
    return bundle.replace('_bundle', '').replace(' 2', '')


def read_bundle_metadata(bundle: str) -> pd.DataFrame:
    prefix = bundle_prefix(bundle)
    csv_path = ROOT / bundle / f'{prefix}.csv'
    df = pd.read_csv(csv_path, encoding='utf-8-sig')
    df.columns = [c.replace('\ufeff', '') for c in df.columns]
    return df.fillna('')


def parse_filename(path: Path) -> Dict[str, object]:
    parts = path.stem.split('_')
    parsed = {
        'source_index_1based': None,
        'query_char_from_filename': '',
        'style_name_from_filename': path.parent.name,
        'author_from_filename': '',
        'work_title_from_filename': '',
    }
    if len(parts) >= 3:
        try:
            parsed['source_index_1based'] = int(parts[0])
        except ValueError:
            parsed['source_index_1based'] = None
        parsed['query_char_from_filename'] = parts[1]
        parsed['style_name_from_filename'] = parts[2]
        if len(parts) >= 4:
            parsed['author_from_filename'] = parts[3]
        if len(parts) >= 5:
            parsed['work_title_from_filename'] = '_'.join(parts[4:])
    return parsed


def collect_records() -> pd.DataFrame:
    records = []
    metadata_by_bundle = {bundle: read_bundle_metadata(bundle) for bundle in BUNDLES}
    for bundle in BUNDLES:
        bundle_dir = ROOT / bundle
        downloads_dirs = sorted(bundle_dir.glob('*_downloads'))
        if not downloads_dirs:
            continue
        downloads_dir = downloads_dirs[0]
        meta = metadata_by_bundle[bundle]
        for style_name, style_label in TARGET_STYLE_LABELS.items():
            style_dir = downloads_dir / style_name
            if not style_dir.exists():
                continue
            for image_path in sorted(p for p in style_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS):
                parsed = parse_filename(image_path)
                source_index = parsed['source_index_1based']
                row = None
                if isinstance(source_index, int):
                    row_index = source_index - 1
                    if 0 <= row_index < len(meta):
                        row = meta.iloc[row_index].to_dict()
                if row is None:
                    row = {}
                records.append({
                    'bundle': bundle,
                    'source_index_1based': source_index,
                    'query_char': row.get('query_char', parsed['query_char_from_filename']),
                    'style_code': row.get('style_code', ''),
                    'style_name': row.get('style_name', parsed['style_name_from_filename']),
                    'style_label': TARGET_STYLE_LABELS.get(str(row.get('style_name', parsed['style_name_from_filename'])), style_label),
                    'era': row.get('era', ''),
                    'author': row.get('author', parsed['author_from_filename']),
                    'work_title': row.get('work_title', parsed['work_title_from_filename']),
                    'detail_title': row.get('detail_title', ''),
                    'image_url': row.get('image_url', ''),
                    'thumb_url': row.get('thumb_url', ''),
                    'image_path': str(image_path.resolve()),
                })
    return pd.DataFrame(records)


def quick_dark_score(image_path: str) -> Tuple[float, float, float]:
    try:
        with Image.open(image_path) as im:
            gray = ImageOps.grayscale(im)
            gray.thumbnail((96, 96), Image.Resampling.BILINEAR)
            arr = np.asarray(gray, dtype=np.float32)
        dark_ratio = float(np.mean(arr < 80))
        bright_ratio = float(np.mean(arr > 200))
        border = border_pixels(arr, 0.08)
        corner = corner_pixels(arr, 0.10)
        return dark_ratio, bright_ratio, float(np.mean(corner)) if corner.size else float(np.mean(border))
    except Exception:
        return 0.0, 0.0, 255.0


def build_sample(records: pd.DataFrame) -> pd.DataFrame:
    if RUN_FULL:
        return records.reset_index(drop=True)

    target = min(SAMPLE_SIZE, len(records))
    base_target = max(0, target - DARK_SUPPLEMENT_SIZE)
    group_cols = ['bundle', 'style_name']
    groups = list(records.groupby(group_cols, sort=True))
    per_group = max(1, base_target // max(1, len(groups)))
    sampled_indices = []
    for _, group in groups:
        n = min(per_group, len(group))
        sampled_indices.extend(group.sample(n=n, random_state=RANDOM_SEED).index.tolist())

    if len(sampled_indices) < base_target:
        remaining = records.drop(index=sampled_indices, errors='ignore')
        need = min(base_target - len(sampled_indices), len(remaining))
        if need > 0:
            sampled_indices.extend(remaining.sample(n=need, random_state=RANDOM_SEED + 1).index.tolist())

    remaining = records.drop(index=sampled_indices, errors='ignore')
    pool_n = min(DARK_POOL_SIZE, len(remaining))
    dark_candidates = []
    if pool_n > 0:
        pool = remaining.sample(n=pool_n, random_state=RANDOM_SEED + 2)
        for idx, row in pool.iterrows():
            dark_ratio, bright_ratio, corner_mean = quick_dark_score(row['image_path'])
            score = dark_ratio + (0.5 if corner_mean < 130 and bright_ratio > 0.01 else 0.0) + bright_ratio * 0.2
            dark_candidates.append((score, idx))
        dark_candidates.sort(reverse=True)
        need = target - len(sampled_indices)
        sampled_indices.extend([idx for _, idx in dark_candidates[:need]])

    if len(sampled_indices) < target:
        remaining = records.drop(index=sampled_indices, errors='ignore')
        need = min(target - len(sampled_indices), len(remaining))
        if need > 0:
            sampled_indices.extend(remaining.sample(n=need, random_state=RANDOM_SEED + 3).index.tolist())

    sampled = records.loc[sampled_indices[:target]].copy()
    return sampled.sample(frac=1, random_state=RANDOM_SEED + 4).reset_index(drop=True)


def border_pixels(arr: np.ndarray, ratio: float = 0.08) -> np.ndarray:
    h, w = arr.shape
    bh = max(1, int(round(h * ratio)))
    bw = max(1, int(round(w * ratio)))
    mask = np.zeros((h, w), dtype=bool)
    mask[:bh, :] = True
    mask[-bh:, :] = True
    mask[:, :bw] = True
    mask[:, -bw:] = True
    return arr[mask]


def corner_pixels(arr: np.ndarray, ratio: float = 0.10) -> np.ndarray:
    h, w = arr.shape
    ch = max(1, int(round(h * ratio)))
    cw = max(1, int(round(w * ratio)))
    return np.concatenate([
        arr[:ch, :cw].ravel(),
        arr[:ch, -cw:].ravel(),
        arr[-ch:, :cw].ravel(),
        arr[-ch:, -cw:].ravel(),
    ])


def center_pixels(arr: np.ndarray, ratio: float = 0.60) -> np.ndarray:
    h, w = arr.shape
    ch = max(1, int(round(h * ratio)))
    cw = max(1, int(round(w * ratio)))
    y0 = max(0, (h - ch) // 2)
    x0 = max(0, (w - cw) // 2)
    return arr[y0:y0 + ch, x0:x0 + cw].ravel()


def component_stats(mask: np.ndarray) -> Dict[str, float]:
    total_area = int(mask.sum())
    if total_area == 0:
        return {
            'component_count': 0,
            'largest_component_area': 0,
            'total_area': 0,
            'largest_component_ratio': 0.0,
        }
    labeled, count = ndimage.label(mask)
    areas = np.bincount(labeled.ravel())
    if len(areas) <= 1:
        largest = 0
        count_nonzero = 0
    else:
        comp_areas = areas[1:]
        # Count tiny specks too, but ignore truly isolated single pixels in the count.
        count_nonzero = int(np.sum(comp_areas >= 2))
        largest = int(comp_areas.max())
    return {
        'component_count': count_nonzero,
        'largest_component_area': largest,
        'total_area': total_area,
        'largest_component_ratio': float(largest / total_area) if total_area else 0.0,
    }


def bbox_stats(mask: np.ndarray) -> Tuple[float, float]:
    ys, xs = np.where(mask)
    if len(xs) == 0 or len(ys) == 0:
        return 0.0, 0.0
    h, w = mask.shape
    bw = int(xs.max() - xs.min() + 1)
    bh = int(ys.max() - ys.min() + 1)
    area_ratio = float((bw * bh) / (w * h))
    aspect = float(max(bw / max(1, bh), bh / max(1, bw)))
    return area_ratio, aspect


def saturation_mean(rgb_arr: np.ndarray) -> float:
    rgb = rgb_arr.astype(np.float32) / 255.0
    maxc = rgb.max(axis=2)
    minc = rgb.min(axis=2)
    sat = np.zeros_like(maxc)
    nonzero = maxc > 1e-6
    sat[nonzero] = (maxc[nonzero] - minc[nonzero]) / maxc[nonzero]
    return float(np.mean(sat))


def has_vector_hint(row: Dict[str, object], image_path: str) -> bool:
    fields = [
        row.get('query_char', ''), row.get('style_name', ''), row.get('style_label', ''),
        row.get('era', ''), row.get('author', ''), row.get('work_title', ''), row.get('detail_title', ''),
        row.get('image_url', ''), row.get('thumb_url', ''), Path(image_path).name,
    ]
    text = ' '.join(str(x) for x in fields)
    return any(hint in text for hint in VECTOR_HINTS)


def is_black_background_candidate_from_features(f: Dict[str, float]) -> bool:
    return bool(
        (f['corner_mean'] < 100 and f['bright_ratio'] > 0.02) or
        (f['border_mean'] < 120 and f['bright_ratio'] > 0.02) or
        (f['dark_ratio'] > 0.50 and f['bright_ratio'] > 0.015) or
        (f['corner_mean'] < f['center_mean'] - 20 and f['corner_mean'] < 130) or
        (f['dark_ratio'] > f['bright_ratio'] * 2 and f['bright_ratio'] > 0.02)
    )


def classify_image(f: Dict[str, float], vector_hint: bool) -> Tuple[str, str, bool]:
    black_candidate = is_black_background_candidate_from_features(f)

    invalid_reasons = []
    if vector_hint:
        invalid_reasons.append('has_vector_hint')
    if f['aspect_ratio'] > 2.8:
        invalid_reasons.append('aspect_ratio_gt_2.8')
    if f['width'] < 20 or f['height'] < 20:
        invalid_reasons.append('too_small')
    if f['dark_bbox_area_ratio'] < 0.02 and f['bright_bbox_area_ratio'] < 0.02:
        invalid_reasons.append('no_content_bbox')
    if f['dark_component_count'] > 120 or f['bright_component_count'] > 120:
        invalid_reasons.append('extreme_component_count')
    if invalid_reasons:
        return 'catalog_or_invalid', ';'.join(invalid_reasons), black_candidate

    if black_candidate:
        if f['largest_bright_component_image_ratio'] > 0.12:
            return 'black_damaged', 'largest_bright_component_image_ratio_gt_0.12', black_candidate
        if f['largest_bright_component_ratio'] > 0.45 and f['largest_bright_component_image_ratio'] > 0.06:
            return 'black_damaged', 'large_bright_component_dominates', black_candidate
        if f['bright_ratio'] > 0.45:
            return 'black_damaged', 'bright_ratio_gt_0.45', black_candidate
        if f['bright_bbox_area_ratio'] > 0.80 and f['bright_component_count'] > 25:
            return 'black_damaged', 'bright_bbox_spans_image_many_components', black_candidate

        if f['bright_component_count'] > 60:
            return 'black_noisy', 'bright_component_count_gt_60', black_candidate
        if f['bright_component_count'] > 35 and f['largest_bright_component_ratio'] < 0.20:
            return 'black_noisy', 'many_fragmented_bright_components', black_candidate
        if f['background_std'] > 55 and f['bright_component_count'] > 30:
            return 'black_noisy', 'noisy_background_many_bright_components', black_candidate
        if 0.02 <= f['bright_ratio'] <= 0.45 and f['bright_component_count'] > 35 and f['largest_bright_component_image_ratio'] < 0.04:
            return 'black_noisy', 'fragmented_small_bright_components', black_candidate

        if f['bright_ratio'] < 0.08 and f['mid_ratio'] > 0.20 and (f['center_mean'] > f['corner_mean'] + 10 or f['p75_gray'] > f['border_mean'] + 15):
            return 'black_gray_ink', 'low_bright_ratio_midtone_gray_ink', black_candidate

        if (
            0.02 <= f['bright_ratio'] <= 0.35 and
            f['bright_component_count'] <= 35 and
            f['largest_bright_component_image_ratio'] <= 0.12 and
            f['largest_bright_component_ratio'] >= 0.15 and
            f['background_std'] <= 55
        ):
            return 'black_clean', 'clean_black_background_candidate', black_candidate

        if f['bright_ratio'] < 0.12 and f['mid_ratio'] > 0.15:
            return 'black_gray_ink', 'fallback_gray_ink_black_background', black_candidate
        return 'black_noisy', 'fallback_black_background_unclean', black_candidate

    if f['dark_component_count'] > 60:
        return 'rubbing_noisy', 'dark_component_count_gt_60', black_candidate
    if f['largest_dark_component_ratio'] < 0.20 and f['dark_component_count'] > 30:
        return 'rubbing_noisy', 'fragmented_dark_components', black_candidate
    if f['background_std'] > 45 and f['dark_component_count'] > 30:
        return 'rubbing_noisy', 'noisy_background_dark_components', black_candidate
    if f['dark_bbox_area_ratio'] > 0.75 and f['dark_component_count'] > 35:
        return 'rubbing_noisy', 'dark_bbox_spans_image_fragmented', black_candidate

    if f['mean_r'] > f['mean_b'] + 10 and f['saturation_mean'] > 0.08 and f['yellow_score'] > 10:
        return 'yellow_paper', 'warm_yellow_background', black_candidate

    if f['border_mean'] > 220 and f['background_std'] < 25 and f['gray_noise_ratio'] < 0.35 and 0.02 <= f['dark_ratio'] <= 0.45:
        return 'clean_white', 'bright_low_noise_background', black_candidate

    if 130 <= f['border_mean'] <= 235 and (f['background_std'] > 25 or f['gray_noise_ratio'] > 0.35) and 0.01 <= f['dark_ratio'] <= 0.60:
        return 'gray_texture', 'gray_or_textured_background', black_candidate

    return 'unknown', 'no_heuristic_match', black_candidate


def profile_one(row: pd.Series, report_index: int) -> Dict[str, object]:
    image_path = row['image_path']
    base = row.to_dict()
    base['global_index'] = report_index
    vector_hint = has_vector_hint(base, image_path)
    try:
        with Image.open(image_path) as im:
            rgb = ImageOps.exif_transpose(im).convert('RGB')
            rgb_arr = np.asarray(rgb, dtype=np.uint8)
            gray_arr = np.asarray(rgb.convert('L'), dtype=np.float32)
    except Exception as exc:
        empty = {
            'image_type': 'catalog_or_invalid',
            'image_subtype_reason': f'read_error:{type(exc).__name__}:{exc}',
            'black_background_candidate': False,
            'has_vector_hint': vector_hint,
        }
        for col in FEATURE_COLUMNS:
            empty.setdefault(col, np.nan)
        return {**base, **empty}

    h, w = gray_arr.shape
    area = float(w * h)
    percentiles = np.percentile(gray_arr, [5, 25, 50, 75, 95])
    border = border_pixels(gray_arr, 0.08)
    corners = corner_pixels(gray_arr, 0.10)
    center = center_pixels(gray_arr, 0.60)

    dark_mask = gray_arr < 120
    bright_mask = gray_arr > 180
    dark = component_stats(dark_mask)
    bright = component_stats(bright_mask)
    dark_bbox_area_ratio, dark_bbox_aspect_ratio = bbox_stats(dark_mask)
    bright_bbox_area_ratio, bright_bbox_aspect_ratio = bbox_stats(bright_mask)

    f = {
        'width': int(w),
        'height': int(h),
        'aspect_ratio': float(max(w / max(1, h), h / max(1, w))),
        'mean_gray': float(np.mean(gray_arr)),
        'median_gray': float(np.median(gray_arr)),
        'std_gray': float(np.std(gray_arr)),
        'p05_gray': float(percentiles[0]),
        'p25_gray': float(percentiles[1]),
        'p50_gray': float(percentiles[2]),
        'p75_gray': float(percentiles[3]),
        'p95_gray': float(percentiles[4]),
        'border_mean': float(np.mean(border)),
        'corner_mean': float(np.mean(corners)),
        'center_mean': float(np.mean(center)),
        'dark_ratio': float(np.mean(gray_arr < 80)),
        'mid_ratio': float(np.mean((gray_arr >= 80) & (gray_arr <= 200))),
        'bright_ratio': float(np.mean(gray_arr > 200)),
        'background_std': float(np.std(border)),
        'gray_noise_ratio': float(np.mean((gray_arr > 120) & (gray_arr < 240))),
        'mean_r': float(np.mean(rgb_arr[:, :, 0])),
        'mean_g': float(np.mean(rgb_arr[:, :, 1])),
        'mean_b': float(np.mean(rgb_arr[:, :, 2])),
        'saturation_mean': saturation_mean(rgb_arr),
        'yellow_score': float(np.mean(rgb_arr[:, :, 0].astype(np.float32)) - np.mean(rgb_arr[:, :, 2].astype(np.float32))),
        'dark_component_count': int(dark['component_count']),
        'largest_dark_component_area': int(dark['largest_component_area']),
        'total_dark_area': int(dark['total_area']),
        'largest_dark_component_ratio': float(dark['largest_component_ratio']),
        'bright_component_count': int(bright['component_count']),
        'largest_bright_component_area': int(bright['largest_component_area']),
        'total_bright_area': int(bright['total_area']),
        'largest_bright_component_ratio': float(bright['largest_component_ratio']),
        'largest_bright_component_image_ratio': float(bright['largest_component_area'] / area) if area else 0.0,
        'dark_bbox_area_ratio': dark_bbox_area_ratio,
        'bright_bbox_area_ratio': bright_bbox_area_ratio,
        'dark_bbox_aspect_ratio': dark_bbox_aspect_ratio,
        'bright_bbox_aspect_ratio': bright_bbox_aspect_ratio,
        'has_vector_hint': bool(vector_hint),
    }
    image_type, reason, black_candidate = classify_image(f, vector_hint)
    return {**base, 'image_type': image_type, 'image_subtype_reason': reason, 'black_background_candidate': black_candidate, **f}


FEATURE_COLUMNS = [
    'width', 'height', 'aspect_ratio', 'mean_gray', 'median_gray', 'std_gray', 'p05_gray', 'p25_gray', 'p50_gray', 'p75_gray', 'p95_gray',
    'border_mean', 'corner_mean', 'center_mean', 'dark_ratio', 'mid_ratio', 'bright_ratio', 'background_std', 'gray_noise_ratio',
    'mean_r', 'mean_g', 'mean_b', 'saturation_mean', 'yellow_score',
    'dark_component_count', 'largest_dark_component_area', 'total_dark_area', 'largest_dark_component_ratio',
    'bright_component_count', 'largest_bright_component_area', 'total_bright_area', 'largest_bright_component_ratio', 'largest_bright_component_image_ratio',
    'dark_bbox_area_ratio', 'bright_bbox_area_ratio', 'dark_bbox_aspect_ratio', 'bright_bbox_aspect_ratio', 'has_vector_hint'
]
REPORT_COLUMNS = [
    'global_index', 'bundle', 'query_char', 'style_code', 'style_name', 'style_label', 'era', 'author', 'work_title', 'detail_title',
    'image_url', 'thumb_url', 'image_path', 'image_type', 'image_subtype_reason',
] + FEATURE_COLUMNS + ['black_background_candidate', 'source_index_1based']


def load_font(size: int = 14):
    candidates = [
        '/System/Library/Fonts/PingFang.ttc',
        '/System/Library/Fonts/STHeiti Light.ttc',
        '/Library/Fonts/Arial Unicode.ttf',
    ]
    for candidate in candidates:
        if Path(candidate).exists():
            try:
                return ImageFont.truetype(candidate, size=size)
            except Exception:
                pass
    return ImageFont.load_default()


def make_contact_sheet(report: pd.DataFrame, image_type: str, out_path: Path, max_samples: int = 80) -> None:
    subset = report[report['image_type'] == image_type]
    if len(subset) > max_samples:
        subset = subset.sample(n=max_samples, random_state=RANDOM_SEED)
    else:
        subset = subset.copy()

    thumb_w, thumb_h = 128, 128
    label_h = 54
    cols = 10
    rows = max(1, math.ceil(len(subset) / cols))
    sheet = Image.new('RGB', (cols * thumb_w, rows * (thumb_h + label_h)), 'white')
    draw = ImageDraw.Draw(sheet)
    font = load_font(13)
    small_font = load_font(11)

    if len(subset) == 0:
        draw.text((12, 12), f'No samples: {image_type}', fill=(0, 0, 0), font=font)
        sheet.save(out_path)
        return

    for pos, (_, row) in enumerate(subset.iterrows()):
        x = (pos % cols) * thumb_w
        y = (pos // cols) * (thumb_h + label_h)
        try:
            with Image.open(row['image_path']) as im:
                im = ImageOps.exif_transpose(im).convert('RGB')
                im.thumbnail((thumb_w, thumb_h), Image.Resampling.LANCZOS)
                canvas = Image.new('RGB', (thumb_w, thumb_h), (245, 245, 245))
                canvas.paste(im, ((thumb_w - im.width) // 2, (thumb_h - im.height) // 2))
        except Exception:
            canvas = Image.new('RGB', (thumb_w, thumb_h), (230, 230, 230))
            ImageDraw.Draw(canvas).text((8, 48), 'READ ERROR', fill=(180, 0, 0), font=font)
        sheet.paste(canvas, (x, y))
        label_lines = [
            f"{row.get('query_char','')} / {row.get('style_label','')}",
            str(row.get('author', ''))[:16],
            str(row.get('image_type', '')),
        ]
        ty = y + thumb_h + 2
        for line in label_lines:
            draw.text((x + 3, ty), line, fill=(0, 0, 0), font=small_font)
            ty += 16
    sheet.save(out_path)


records = collect_records()
print(f'Total target images found: {len(records)}')
print(records.groupby(['bundle', 'style_name']).size().unstack(fill_value=0))

sample = build_sample(records)
print(f'Profiling sample size: {len(sample)}')

rows = []
for i, (_, row) in enumerate(sample.iterrows()):
    rows.append(profile_one(row, i))
    if (i + 1) % 50 == 0:
        print(f'profiled {i + 1}/{len(sample)}')

report = pd.DataFrame(rows)
for col in REPORT_COLUMNS:
    if col not in report.columns:
        report[col] = np.nan
report = report[REPORT_COLUMNS]
report_path = OUTPUT_DIR / 'image_type_report.csv'
report.to_csv(report_path, index=False, encoding='utf-8-sig')
print(f'Wrote: {report_path}')

for image_type in IMAGE_TYPES:
    out_path = SAMPLES_DIR / f'{image_type}.png'
    make_contact_sheet(report, image_type, out_path, max_samples=80)
    print(f'Wrote: {out_path}')

print('\nImage type counts:')
print(report['image_type'].value_counts().reindex(IMAGE_TYPES, fill_value=0))
print('\nBlack background candidate count:', int(report['black_background_candidate'].sum()))
print('\nBlack subtypes:')
print(report[report['image_type'].str.startswith('black_', na=False)]['image_type'].value_counts().reindex(['black_clean','black_gray_ink','black_damaged','black_noisy'], fill_value=0))
print('\nUnknown count:', int((report['image_type'] == 'unknown').sum()))


Total target images found: 408358
style_name               楷书    章草     篆书     草书     行书    隶书
bundle                                                      
common_c_bundle       13320  3831   5575  13568  20389  6003
common_g_bundle 2     18506  4845   8423  17041  27993  8795
common_h_bundle       13664  4046   7584  14897  22873  6483
common_i_bundle       18802  4377  10118  17499  26722  7966
common_j_bundle       16496  4031   8748  14614  24024  7425
less_common_b_bundle   5543  2109   3715   7079   8322  2932
Profiling sample size: 300
profiled 50/300
profiled 100/300
profiled 150/300
profiled 200/300
profiled 250/300
profiled 300/300
Wrote: /Users/lainantian/Developer/Caligraphy/image_type_profile/image_type_report.csv
Wrote: /Users/lainantian/Developer/Caligraphy/image_type_profile/samples/clean_white.png
Wrote: /Users/lainantian/Developer/Caligraphy/image_type_profile/samples/yellow_paper.png
Wrote: /Users/lainantian/Developer/Caligraphy/image_type_profile/samples/gray_textur